# Session 4 — Analysis: Static vs Dynamic Computation Graph

## Background

Sessions 1–3 used a fully static forward pass: the computation graph is identical on every step, which is what XLA requires to fuse operations and avoid recompilation. Real research often involves models with data-dependent control flow — sparse attention patterns, conditional skip connections, variable-length routing. On GPU, PyTorch's eager execution handles these without penalty. On TPU, a Python `if` statement whose condition depends on a tensor value forces XLA to either retrace and recompile the graph or fall back to slower interpreted execution. This session isolates that cost directly.

## Goal

At the end of this session, participants will understand why dynamic control flow is the primary constraint on TPU suitability for certain research workloads, will have measured the throughput penalty of a data-dependent branch on each device, and will be able to refactor a branching `forward()` into a compilable equivalent.

## Question

How much throughput does a Python branch in `forward()` cost on a compiled (TPU) vs eager (GPU) device, and does a tensor-valued equivalent recover it?

---

**Prerequisites:** Run `01-benchmark-gpu.ipynb` and `02-benchmark-tpu.ipynb` first to populate `results/gpu_graph_variants.json` and `results/tpu_graph_variants.json`.

In [1]:
import json, os

VARIANT_ORDER = ["StaticFF", "DynamicFF", "StaticEquivalentFF"]

def load_variant_results(results_dir="results"):
    out = {}
    for fname in ["gpu_graph_variants.json", "tpu_graph_variants.json"]:
        path = os.path.join(results_dir, fname)
        if not os.path.exists(path):
            print(f"Missing: {path} — run the corresponding benchmark notebook first.")
            continue
        with open(path) as f:
            data = json.load(f)
        hw = data["hardware"]
        out[hw] = {
            "device_name": data.get("device_name", ""),
            "batch_size":  data.get("batch_size"),
            "seq_len":     data.get("seq_len"),
            "results":     data["results"],
        }
    return out

data = load_variant_results("results")

print("Loaded:")
for hw, d in data.items():
    print(f"  {hw} ({d['device_name']}) — batch={d['batch_size']}, seq_len={d['seq_len']}")
    for v, r in d["results"].items():
        if isinstance(r, dict):
            print(f"    {v:<22}: {r['throughput']:,.0f} samples/sec  ({r['latency_ms']:.1f} ms/step)")

Loaded:
  GPU (NVIDIA L4) — batch=256, seq_len=128
    StaticFF              : 2,038 samples/sec  (125.6 ms/step)
    DynamicFF             : 1,199 samples/sec  (213.5 ms/step)
    StaticEquivalentFF    : 1,144 samples/sec  (223.7 ms/step)
    PaddingMask           : 1,162 samples/sec  (220.2 ms/step)
    EarlyExitDynamic      : 1,038 samples/sec  (246.6 ms/step)
    EarlyExitStatic       : 603 samples/sec  (424.4 ms/step)
    FixedLoopFF           : 407 samples/sec  (629.5 ms/step)
    VariableLoopFF        : 806 samples/sec  (317.6 ms/step)
  TPU (v5litepod-1) — batch=256, seq_len=128
    StaticFF              : 24,017 samples/sec  (10.7 ms/step)
    DynamicFF             : 9,453 samples/sec  (27.1 ms/step)
    StaticEquivalentFF    : 23,806 samples/sec  (10.8 ms/step)
    PaddingMask           : 23,960 samples/sec  (10.7 ms/step)
    EarlyExitDynamic      : 7,522 samples/sec  (34.0 ms/step)
    EarlyExitStatic       : 18,577 samples/sec  (13.8 ms/step)
    FixedLoopFF           : 15

In [2]:
hw_order = [hw for hw in ["GPU", "TPU"] if hw in data]

col = 24
header = f"{'Variant':<22}" + "".join(f" | {hw + ' (samples/s)':>{col}}" for hw in hw_order)
if len(hw_order) == 2:
    header += f" | {'Dynamic penalty (TPU)':>{col}}"
print(header)
print("-" * len(header))

for v in VARIANT_ORDER:
    row = f"{v:<22}"
    for hw in hw_order:
        r = data[hw]["results"].get(v)
        if isinstance(r, dict):
            row += f" | {f"{r['throughput']:,.0f}":>{col}}"
        else:
            row += f" | {'—':>{col}}"
    if len(hw_order) == 2 and v == "DynamicFF" and "TPU" in data:
        tpu_static  = data["TPU"]["results"].get("StaticFF", {})
        tpu_dynamic = data["TPU"]["results"].get("DynamicFF", {})
        s = tpu_static.get("throughput") if isinstance(tpu_static, dict) else None
        d_ = tpu_dynamic.get("throughput") if isinstance(tpu_dynamic, dict) else None
        if s and d_:
            row += f" | {d_/s:>{col}.3f}× vs Static"
        else:
            row += f" | {'—':>{col}}"
    elif len(hw_order) == 2:
        row += f" | {'—':>{col}}"
    print(row)

if "TPU" in data:
    s  = data["TPU"]["results"].get("StaticFF", {}).get("throughput")
    d_ = data["TPU"]["results"].get("DynamicFF", {}).get("throughput")
    eq = data["TPU"]["results"].get("StaticEquivalentFF", {}).get("throughput")
    if s and d_ and eq:
        print(f"\nTPU penalty  (Dynamic / Static)      : {d_/s:.3f}×")
        print(f"TPU recovery (StaticEquiv / Static)  : {eq/s:.3f}×")
        print(f"Penalty recovered                    : {(eq-d_)/(s-d_)*100:.1f}%" if s != d_ else "n/a")

Variant                |          GPU (samples/s) |          TPU (samples/s) |    Dynamic penalty (TPU)
-------------------------------------------------------------------------------------------------------
StaticFF               |                    2,038 |                   24,017 |                        —
DynamicFF              |                    1,199 |                    9,453 |                    0.394× vs Static
StaticEquivalentFF     |                    1,144 |                   23,806 |                        —

TPU penalty  (Dynamic / Static)      : 0.394×
TPU recovery (StaticEquiv / Static)  : 0.991×
Penalty recovered                    : 98.5%


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os

hw_order = [hw for hw in ["GPU", "TPU"] if hw in data]
COLORS   = {"GPU": "#76b900", "TPU": "#4285F4"}

x      = np.arange(len(VARIANT_ORDER))
width  = 0.35
offset = [-width/2, width/2]

fig, ax = plt.subplots(figsize=(10, 5))

for i, hw in enumerate(hw_order):
    values = [
        (data[hw]["results"].get(v) or {}).get("throughput", 0)
        for v in VARIANT_ORDER
    ]
    bars = ax.bar(x + offset[i], values, width, label=hw, color=COLORS[hw], alpha=0.85)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                    f"{val:,.0f}", ha="center", va="bottom", fontsize=8, rotation=0)

ax.set_xticks(x)
ax.set_xticklabels(VARIANT_ORDER, fontsize=11)
ax.set_ylabel("Throughput (samples / second)", fontsize=12)
ax.set_title("Static vs Dynamic Graph — GPU and TPU (batch=256, seq_len=128)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
import matplotlib.ticker as mticker
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()

os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_4_chart_variants.png", dpi=150)
print("Saved session_4_chart_variants.png")

Saved session_4_chart_variants.png


In [4]:
hw_order = [hw for hw in ["GPU", "TPU"] if hw in data]

if hw_order:
    fig, axes = plt.subplots(1, len(hw_order), figsize=(11, 5))
    if len(hw_order) == 1:
        axes = [axes]

    COLORS_HW = {"GPU": "#76b900", "TPU": "#4285F4"}
    HIGHLIGHT  = {"GPU": None, "TPU": "DynamicFF"}

    for ax, hw in zip(axes, hw_order):
        values = [
            (data[hw]["results"].get(v) or {}).get("latency_ms", 0)
            for v in VARIANT_ORDER
        ]
        bar_colors = []
        for v, val in zip(VARIANT_ORDER, values):
            if v == HIGHLIGHT[hw] and val > 0:
                bar_colors.append("#e74c3c")
            else:
                bar_colors.append(COLORS_HW[hw])

        bars = ax.bar(VARIANT_ORDER, values, color=bar_colors, alpha=0.85, width=0.5)
        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + max(values) * 0.01,
                        f"{val:.1f} ms", ha="center", va="bottom", fontsize=9)

        ax.set_ylabel("Latency (ms / step)", fontsize=11)
        ax.set_title(f"{hw}  ({data[hw]['device_name']})", fontsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_ylim(bottom=0)
        ax.set_xticks(range(len(VARIANT_ORDER)))
        ax.set_xticklabels(VARIANT_ORDER, fontsize=9, rotation=10, ha="right")

    fig.suptitle("Per-Step Latency — DynamicFF adds 2.5× TPU step time", fontsize=13)
    plt.tight_layout()

    plt.savefig("../docs/assets/session_4_chart_latency.png", dpi=150)
    print("Saved session_4_chart_latency.png")

Saved session_4_chart_latency.png


## Interpreting the results

| Observation | What it means |
|---|---|
| GPU throughput within ~5% across all three variants | Eager dispatch evaluates `out.mean()` immediately — no graph structure penalty |
| TPU throughput drops on `DynamicFF` | `if out.mean() > 0` forces XLA to sync and materialize the tensor to CPU, breaking lazy evaluation |
| `StaticEquivalentFF` recovers most/all TPU performance | `torch.where` is a tensor op — the condition stays in the XLA graph; no sync required |

## The refactoring pattern

```python
# Before — data-dependent Python branch (XLA-hostile)
if out.mean() > 0:
    return out
else:
    return -out

# After — tensor-valued equivalent (XLA-compilable)
return torch.where(out.mean() > 0, out, -out)
```

The takeaway is not "TPUs are fragile" — it is that **the constraint is refactorable**. Many
data-dependent branches in research code can be replaced with `torch.where`, `torch.masked_fill`,
or sparse masking without changing model behaviour. When they cannot be refactored (e.g.,
variable-length sequence routing, per-sample sparse attention with dynamic sparsity patterns),
the GPU remains the correct choice.

## Decision guidance

- **Refactor and use TPU** if the branch condition is derivable from a tensor operation (`mean`, `max`, `norm`, mask thresholding).
- **Stay on GPU** if the branch depends on runtime logic that cannot be expressed as a tensor op (variable-length inputs, per-sample routing, dynamic sparsity).
- The **Realistic Scenarios** cells in the benchmark notebooks show where each pattern falls on this spectrum.

In [5]:

# Part B — Realistic Scenarios chart
# All 5 Part B variants plus the StaticFF baseline for reference

PART_B_ORDER = [
    "StaticFF",
    "PaddingMask",
    "EarlyExitDynamic",
    "EarlyExitStatic",
    "FixedLoopFF",
    "VariableLoopFF",
]
PART_B_LABELS = [
    "StaticFF\n(baseline)",
    "B1: PaddingMask",
    "B2: EarlyExitDynamic",
    "B2: EarlyExitStatic",
    "B3: FixedLoopFF",
    "B3: VariableLoopFF",
]

hw_order = [hw for hw in ["GPU", "TPU"] if hw in data]
COLORS   = {"GPU": "#76b900", "TPU": "#4285F4"}

x      = np.arange(len(PART_B_ORDER))
width  = 0.35
offset = [-width/2, width/2]

fig, ax = plt.subplots(figsize=(13, 5))

for i, hw in enumerate(hw_order):
    values = [
        (data[hw]["results"].get(v) or {}).get("throughput", 0)
        for v in PART_B_ORDER
    ]
    bars = ax.bar(x + offset[i], values, width, label=hw, color=COLORS[hw], alpha=0.85)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 200,
                    f"{val:,.0f}", ha="center", va="bottom", fontsize=7.5, rotation=0)

ax.set_xticks(x)
ax.set_xticklabels(PART_B_LABELS, fontsize=9.5)
ax.set_ylabel("Throughput (samples / second)", fontsize=12)
ax.set_title(
    "Realistic Scenarios — Throughput by Variant, GPU and TPU (batch=256, seq_len=128)",
    fontsize=12,
)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
import matplotlib.ticker as mticker
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# shade StaticFF as reference
ax.axvspan(-0.5, 0.5, color="gray", alpha=0.07, label="_nolegend_")

plt.tight_layout()
plt.savefig("../docs/assets/session_4_chart_scenarios.png", dpi=150)
print("Saved session_4_chart_scenarios.png")


Saved session_4_chart_scenarios.png
